In [0]:
from pyspark.sql import functions as F
from functools import reduce
import re

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.bronze")

In [0]:
bucket = "s3://ro-company-lake"

mfinante_sf_path = f"{bucket}/raw_v2/mfinante/situatii_financiare_uu/"

display(dbutils.fs.ls(mfinante_sf_path))

In [0]:
def list_all_files(path):
    all_files = []

    def recurse(p):
        for item in dbutils.fs.ls(p):
            if item.isDir():
                recurse(item.path)
            else:
                all_files.append(item.path)

    recurse(path)
    return all_files


all_files = list_all_files(mfinante_sf_path)

print("Total files found:", len(all_files))

for f in sorted(all_files):
    print(f)

In [0]:
def extract_year(path):
    match = re.search(r"web_uu_an(\d{4})\.(txt|csv)$", path, re.IGNORECASE)
    return int(match.group(1)) if match else None


web_uu_txt_files = [
    f for f in all_files
    if re.search(r"web_uu_an\d{4}\.txt$", f, re.IGNORECASE)
]

web_uu_csv_files = [
    f for f in all_files
    if re.search(r"web_uu_an\d{4}\.csv$", f, re.IGNORECASE)
]

txt_by_year = {
    extract_year(f): f
    for f in web_uu_txt_files
    if extract_year(f) is not None
}

csv_by_year = {
    extract_year(f): f
    for f in web_uu_csv_files
    if extract_year(f) is not None
}

available_years = sorted(
    set(txt_by_year.keys()).intersection(set(csv_by_year.keys()))
)

print("TXT years:", sorted(txt_by_year.keys()))
print("CSV years:", sorted(csv_by_year.keys()))
print("Available years:", available_years)

In [0]:
candidate_separators = ["^", ";", "|", "\t", ","]


def detect_separator(path):
    preview = spark.read.text(path)
    first_line = preview.limit(1).collect()[0]["value"]

    counts = {
        sep: first_line.count(sep)
        for sep in candidate_separators
    }

    detected = max(counts, key=counts.get)

    print("Path:", path)
    print("Detected separator:", repr(detected))
    print("Counts:", counts)

    return detected

In [0]:
raw_dfs = []

for year in available_years:
    txt_path = txt_by_year[year]

    print(f"\nLoading financial data for year {year}")
    print(txt_path)

    data_sep = detect_separator(txt_path)

    raw_df = (
        spark.read
        .option("header", False)
        .option("inferSchema", False)
        .option("sep", data_sep)
        .option("encoding", "UTF-8")
        .csv(txt_path)
        .withColumn("_source_year", F.lit(year))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    raw_dfs.append(raw_df)


if not raw_dfs:
    raise ValueError("No WEB_UU TXT files found.")

mfinante_uu_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    raw_dfs
)

display(
    mfinante_uu_all
    .groupBy("_source_year")
    .count()
    .orderBy("_source_year")
)

In [0]:
display(mfinante_uu_all.limit(20))

print("Rows:", mfinante_uu_all.count())
print("Columns:", mfinante_uu_all.columns)
print("Number of columns:", len(mfinante_uu_all.columns))

In [0]:
(
    mfinante_uu_all
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.bronze.mfinante_uu_raw")
)

In [0]:
schema_dfs = []

for year in available_years:
    csv_path = csv_by_year[year]

    print(f"\nLoading schema file for year {year}")
    print(csv_path)

    schema_df = (
        spark.read
        .text(csv_path)
        .withColumn("_source_year", F.lit(year))
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
    )

    schema_dfs.append(schema_df)


if not schema_dfs:
    raise ValueError("No WEB_UU CSV schema files found.")

mfinante_uu_schema_all = reduce(
    lambda a, b: a.unionByName(b, allowMissingColumns=True),
    schema_dfs
)

display(
    mfinante_uu_schema_all
    .groupBy("_source_year")
    .count()
    .orderBy("_source_year")
)

In [0]:
display(mfinante_uu_schema_all.orderBy("_source_year").limit(100))

print("Rows:", mfinante_uu_schema_all.count())
print("Columns:", mfinante_uu_schema_all.columns)

In [0]:
(
    mfinante_uu_schema_all
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("company_ro.bronze.mfinante_uu_schema_raw")
)

In [0]:
display(spark.sql("""
SELECT
  _source_year,
  COUNT(*) AS rows,
  COUNT(DISTINCT _source_file) AS source_files
FROM company_ro.bronze.mfinante_uu_raw
GROUP BY _source_year
ORDER BY _source_year
"""))

In [0]:
display(spark.sql("""
SELECT
  _source_year,
  COUNT(*) AS schema_rows,
  COUNT(DISTINCT _source_file) AS source_files
FROM company_ro.bronze.mfinante_uu_schema_raw
GROUP BY _source_year
ORDER BY _source_year
"""))

In [0]:
display(spark.sql("SHOW TABLES IN company_ro.bronze"))

In [0]:
display(spark.sql("""
SELECT
  MIN(_source_year) AS min_year,
  MAX(_source_year) AS max_year,
  COUNT(DISTINCT _source_year) AS number_of_years,
  COUNT(*) AS total_rows
FROM company_ro.bronze.mfinante_uu_raw
"""))